In [1]:
from multitudcsd.config import get_spark_session
from multitudcsd.storage import read_delta, get_table_path

In [2]:
spark = get_spark_session()

In [39]:
print(spark.sparkContext._jvm.org.apache.hadoop.util.NativeCodeLoader.isNativeCodeLoaded())

True


In [4]:
a = get_table_path("bronze","bronze_gtfs_tripupdates")
print(a)

D:/05_MasterUCM/TFM/multitudcsd/data/lakehouse/bronze/bronze_gtfs_tripupdates


In [5]:
import os
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("hadoop:", spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion())
print("nativa cargada:", spark.sparkContext._jvm.org.apache.hadoop.util.NativeCodeLoader.isNativeCodeLoaded())

HADOOP_HOME: C:\Hadoop
hadoop: 3.3.4
nativa cargada: True


In [7]:
from pathlib import Path
print(sorted(p.name for p in Path(r"C:\Hadoop\bin").iterdir()))

jvm = spark.sparkContext._jvm
print(jvm.java.lang.System.getProperty("java.library.path"))

# Esto lanza el error real, que NativeCodeLoader se traga en un LOG.debug.
jvm.java.lang.System.loadLibrary("hadoop")

['hadoop.dll', 'winutils.exe']
C:\Program Files\Java\jdk-17\bin;C:\WINDOWS\Sun\Java\bin;C:\WINDOWS\system32;C:\WINDOWS;C:\Hadoop\bin;D:\05_MasterUCM\TFM\multitudcsd\.venv\Scripts;C:\Program Files\Common Files\Oracle\Java\javapath;C:\Program Files (x86)\Common Files\Oracle\Java\javapath;C:\Program Files (x86)\Intel\iCLS Client\;C:\Program Files\Intel\iCLS Client\;C:\windows\system32;C:\windows;C:\windows\System32\Wbem;C:\windows\System32\WindowsPowerShell\v1.0\;C:\Program Files (x86)\Intel\Intel(R) Management Engine Components\DAL;C:\Program Files\Intel\Intel(R) Management Engine Components\DAL;C:\Program Files (x86)\Intel\Intel(R) Management Engine Components\IPT;C:\Program Files\Intel\Intel(R) Management Engine Components\IPT;C:\Program Files (x86)\NVIDIA Corporation\PhysX\Common;C:\WINDOWS\system32;C:\WINDOWS;C:\WINDOWS\System32\Wbem;C:\WINDOWS\System32\WindowsPowerShell\v1.0\;C:\WINDOWS\System32\OpenSSH\;C:\Program Files\Intel\WiFi\bin\;C:\Program Files\Common Files\Intel\WirelessCo

In [3]:
#comprobando que hay datos en las tablas antes de hacer silver y gold
for tabla in ["bronze_gtfs_tripupdates", "bronze_nextbike_status",
              "bronze_viz_disruptions", "bronze_gtfs_static_stops",
              "bronze_gtfs_static_routes", "bronze_gtfs_static_stop_times",
              "bronze_gtfs_static_trips", "bronze_nextbike_station_information"]:
    df = read_delta(spark, "bronze", tabla)
    print(tabla, df.count())

bronze_gtfs_tripupdates 92701
bronze_nextbike_status 14685
bronze_viz_disruptions 361
bronze_gtfs_static_stops 3005
bronze_gtfs_static_routes 93
bronze_gtfs_static_stop_times 501537
bronze_gtfs_static_trips 61284
bronze_nextbike_station_information 1049


In [13]:
read_delta(spark, "bronze", "bronze_nextbike_station_information").show(3, truncate=80)

+----------+--------------------------------------------------------------------------------+------------------------------------------------------------------------------+-----------------+------+--------------------------+-----------+
|station_id|                                                                    payload_json|                                                                    source_url|feed_last_updated|source|                 ingest_ts|ingest_date|
+----------+--------------------------------------------------------------------------------+------------------------------------------------------------------------------+-----------------+------+--------------------------+-----------+
| 116227173|{"station_id": "116227173", "name": "EDEKA Wiesbadener Straße", "short_name":...|https://gbfs.nextbike.net/maps/gbfs/v2/nextbike_bn/en/station_information.json|       1788299239|  real|2026-09-01 23:48:18.800353| 2026-09-01|
| 116227471|{"station_id": "116227471", "name": "EDE

In [14]:
from multitudcsd.transforms.geo import compute_h3_cell

# Puerta de Brandeburgo, punto de referencia del recorrido del CSD
celda = compute_h3_cell(52.5163, 13.3777, resolution=9)
print(celda)

891f1d48863ffff


In [15]:
bronze_info.select("payload_json").show(1, truncate=False)

NameError: name 'bronze_info' is not defined

In [17]:
def _inspecciona_un_payload_de_viz(spark):
    from multitudcsd.storage import read_delta
    df = read_delta(spark, "bronze", "bronze_viz_disruptions")
    df.select("payload_json").show(5, truncate=False)

In [18]:
print(_inspecciona_un_payload_de_viz(spark))

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [19]:
silver_bikes = read_delta(spark, "silver", "silver_bike_availability")
silver_bikes.select("station_id", "h3_index", "num_bikes_available", "reading_ts").show(5)

silver_delays = read_delta(spark, "silver", "silver_transit_delays")
silver_delays.select("route_id", "stop_id", "h3_index", "delay_seconds", "feed_ts").show(5)

silver_disruptions = read_delta(spark, "silver", "silver_disruptions")
silver_disruptions.select(
    "disruption_id", "h3_index", "street", "severity", "valid_from", "valid_to"
).show(5, truncate=40)

+----------+---------------+-------------------+-------------------+
|station_id|       h3_index|num_bikes_available|         reading_ts|
+----------+---------------+-------------------+-------------------+
| 167312062|891f1d48e9bffff|                  3|2026-08-23 23:15:00|
| 167339040|891f1d4d063ffff|                  1|2026-08-23 23:15:00|
| 167369847|891f1d4d10bffff|                  2|2026-08-23 23:15:00|
| 167375258|891f1d4d1cbffff|                  0|2026-08-23 23:15:00|
| 167424125|891f1d4f65bffff|                  1|2026-08-23 23:15:00|
+----------+---------------+-------------------+-------------------+
only showing top 5 rows

+--------+--------------------+--------+-------------+-------------------+
|route_id|             stop_id|h3_index|delay_seconds|            feed_ts|
+--------+--------------------+--------+-------------+-------------------+
|1928_700|de:12063:90021000...|    NULL|            0|2026-08-23 23:53:50|
|1928_700|de:12063:90021020...|    NULL|            0|

In [22]:
for tabla in ["silver_bike_availability", "silver_disruptions", "silver_transit_delays"]:
    df = read_delta(spark, "silver", tabla)
    print(tabla, df.count())

silver_bike_availability 3146
silver_disruptions 1
silver_transit_delays 53032


In [17]:
silver_bikes = read_delta(spark, "silver", "silver_bike_availability")
silver_bikes.select("station_id", "h3_index", "num_bikes_available", "reading_ts").show(5)


+----------+---------------+-------------------+-------------------+
|station_id|       h3_index|num_bikes_available|         reading_ts|
+----------+---------------+-------------------+-------------------+
| 167312062|891f1d48e9bffff|                  3|2026-08-23 23:15:00|
| 167339040|891f1d4d063ffff|                  1|2026-08-23 23:15:00|
| 167369847|891f1d4d10bffff|                  2|2026-08-23 23:15:00|
| 167375258|891f1d4d1cbffff|                  0|2026-08-23 23:15:00|
| 167424125|891f1d4f65bffff|                  1|2026-08-23 23:15:00|
+----------+---------------+-------------------+-------------------+
only showing top 5 rows



In [26]:
silver_delays = read_delta(spark, "silver", "silver_transit_delays")
silver_delays.select("route_id", "stop_id", "h3_index", "delay_seconds", "feed_ts").show(50)


+---------+--------------------+---------------+-------------+-------------------+
| route_id|             stop_id|       h3_index|delay_seconds|            feed_ts|
+---------+--------------------+---------------+-------------+-------------------+
| 1928_700|de:12063:90021000...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021020...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021020...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021020...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021021...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021024...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021032...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021050...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021050...|           NULL|            0|2026-08-23 23:53:50|
| 19

In [6]:
silver_disruptions = read_delta(spark, "silver", "silver_disruptions")
silver_disruptions.select(
    "disruption_id", "h3_index", "street", "severity", "valid_from", "valid_to"
).show(5, truncate=40)

+--------------------------+---------------+----------------------------------------+--------+----------+--------+
|             disruption_id|       h3_index|                                  street|severity|valid_from|valid_to|
+--------------------------+---------------+----------------------------------------+--------+----------+--------+
|AdbNO/r_AdbNO/16102_AdB-NO|891f1884397ffff|A115 Dreieck Nuthetal Richtung Dreiec...|    NULL|      NULL|    NULL|
|AdbNO/r_AdbNO/16185_AdB-NO|891f188453bffff|A115 AVUS, Dreieck Funkturm Richtung ...|    NULL|      NULL|    NULL|
|AdbNO/r_AdbNO/16186_AdB-NO|891f188453bffff|A115 AVUS, Dreieck Funkturm Richtung ...|    NULL|      NULL|    NULL|
|AdbNO/r_AdbNO/16197_AdB-NO|891f1d48227ffff|A100 Stadtring Berlin, Wedding Richtu...|    NULL|      NULL|    NULL|
|AdbNO/r_AdbNO/16211_AdB-NO|891f1d49cc7ffff|A100 Stadtring Berlin, Wilmersdorf Ri...|    NULL|      NULL|    NULL|
+--------------------------+---------------+------------------------------------

In [25]:
from pyspark.sql import functions as F
gold_mp = read_delta(spark, "gold", "gold_mobility_pressure")
gold_mp.orderBy(F.desc("num_lecturas_bici")).show(10)

gold_reliability = read_delta(spark, "gold", "gold_line_reliability")
gold_reliability.orderBy(F.desc("avg_delay_seconds")).show(10)

gold_disruptions = read_delta(spark, "gold", "gold_disruptions_by_cell")
gold_disruptions.orderBy(F.desc("num_disruptions")).show(10)

+---------------+-----------+-------------------+-------------------+-----------------+------------------+------------------+---------------------------+
|       h3_index|hour_of_day|avg_bikes_available|avg_docks_available|num_lecturas_bici| avg_delay_seconds|       pct_on_time|num_actualizaciones_retraso|
+---------------+-----------+-------------------+-------------------+-----------------+------------------+------------------+---------------------------+
|891f1d48b13ffff|         23| 0.3333333333333333|  5.833333333333333|               12|               7.5|            0.9375|                         16|
|891f1d48c13ffff|         23| 0.8333333333333334| 3.1666666666666665|               12|              NULL|              NULL|                       NULL|
|891f1d48903ffff|         23|                2.9|                4.4|               10|              NULL|              NULL|                       NULL|
|891f1d49b77ffff|         23|                1.7|                3.4|       

In [8]:
for tabla in ["gold_disruptions_by_cell", "gold_line_reliability", "gold_mobility_pressure"]:
    df = read_delta(spark, "gold", tabla)
    print(tabla, df.count())

gold_disruptions_by_cell 73
gold_line_reliability 1560
gold_mobility_pressure 3428


In [10]:
from pyspark.sql import functions as F
gold_mp = read_delta(spark, "gold", "gold_mobility_pressure")
gold_mp.orderBy(F.desc("num_lecturas_bici")).show(10)

+---------------+-----------+-------------------+-------------------+-----------------+------------------+------------------+---------------------------+
|       h3_index|hour_of_day|avg_bikes_available|avg_docks_available|num_lecturas_bici| avg_delay_seconds|       pct_on_time|num_actualizaciones_retraso|
+---------------+-----------+-------------------+-------------------+-----------------+------------------+------------------+---------------------------+
|891f1d48b13ffff|         20| 1.6666666666666667|                4.5|               42|  71.3157894736842|0.8157894736842105|                        228|
|891f1d48c13ffff|         20|0.16666666666666666| 3.8333333333333335|               42|              NULL|              NULL|                       NULL|
|891f1d48903ffff|         20|  2.914285714285714|  4.285714285714286|               35|              NULL|              NULL|                       NULL|
|891f1d49b77ffff|         20|                1.6|                3.0|       

In [11]:
gold_reliability = read_delta(spark, "gold", "gold_line_reliability")
gold_reliability.orderBy(F.desc("avg_delay_seconds")).show(10)


+---------+-----------+------------------+-------------------+-------------------+
| route_id|hour_of_day| avg_delay_seconds|        pct_on_time|num_actualizaciones|
+---------+-----------+------------------+-------------------+-------------------+
|18347_700|         19|            7591.0|                0.0|                 48|
|18347_700|         20|            7591.0|                0.0|                 12|
|19063_100|         20|6087.0329670329675|                0.0|                 91|
| 5635_900|          1|            4298.0|                0.0|                  2|
|19063_100|         19|2808.4615384615386|                0.0|                 52|
|20969_700|          0|2612.3333333333335|0.16666666666666666|                  6|
|24057_100|          1|            2460.0|                0.0|                  2|
|19156_109|         20|2357.1428571428573| 0.2857142857142857|                 84|
| 4520_700|         19|2310.9166666666665|                0.0|                 48|
|191

In [12]:
gold_disruptions = read_delta(spark, "gold", "gold_disruptions_by_cell")
gold_disruptions.orderBy(F.desc("num_disruptions")).show(10)

+---------------+---------------+
|       h3_index|num_disruptions|
+---------------+---------------+
|891f1d4f2c3ffff|              2|
|891f1d489dbffff|              2|
|891f1d4f3bbffff|              2|
|891f1d48c2bffff|              2|
|891f1d49dcbffff|              2|
|891f1d48877ffff|              2|
|891f1d4997bffff|              1|
|891f1d48ba3ffff|              1|
|891f1d4ab2bffff|              1|
|891f1d4f667ffff|              1|
+---------------+---------------+
only showing top 10 rows



In [27]:
# explorar las tablas con sql como delta tables

In [5]:
from multitudcsd.config import get_spark_session
from multitudcsd.storage import read_delta

read_delta(spark, "silver", "silver_disruptions").createOrReplaceTempView("disruptions")

spark.sql("""
    SELECT * from disruptions
""").show(truncate=False)

+--------------------------------+---------+--------+-------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------+-------------------+-------------------+----------+----------+---------------+
|disruption_id                   |subtype  |severity|street                                                                                                             |section                                                                                                            |content                                                                                                                                                    |ob

In [31]:
##la tabla estaba cogiendo mal el id y tengo que resetearla

#from multitudcsd.config import get_spark_session
#from multitudcsd.storage import get_table_path

#spark = get_spark_session("mantenimiento")
#ruta = get_table_path("bronze", "bronze_viz_disruptions")

#spark.sql(f"DELETE FROM delta.`{ruta}`")
#spark.sql(f"SELECT COUNT(*) FROM delta.`{ruta}`").show()

+--------+
|count(1)|
+--------+
|       0|
+--------+

